# Phase 2: 피처 엔지니어링 (Feature Engineering)

## 이 노트북에서 하는 것
원본 OHLCV 데이터에서 모델이 학습할 수 있는 **의미있는 특성(Feature)**들을 만듭니다.

### 왜 피처 엔지니어링이 중요한가?
원본 가격 데이터만으로는 "내일 가격이 오를지 내릴지" 판단이 어렵습니다.  
트레이더들이 수십년간 써온 **기술적 지표(Technical Indicators)**를 수치화하면  
모델이 패턴을 훨씬 쉽게 학습합니다.

### 추가할 지표 목록
| 카테고리 | 지표 | 용도 |
|----------|------|------|
| 추세 | SMA(20,50), EMA(12,26) | 중장기 추세 방향 |
| 모멘텀 | RSI(14), MACD | 추세 강도, 전환 신호 |
| 변동성 | Bollinger Bands, ATR | **고가/저가 예측의 핵심** |
| 거래량 | OBV, Volume_ratio | 매수/매도 압력 |
| 가격패턴 | 전일대비 변화율, Lag 특성 | 단기 모멘텀 |

In [ ]:
# ============================================================
# 셀 1: 패키지 설치 및 데이터 로드
# ============================================================
!pip install pandas-ta yfinance plotly -q

import pandas as pd
import numpy as np
import pandas_ta as ta
import matplotlib.pyplot as plt
import warnings
import os

warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (14, 6)

# Phase 1에서 저장한 데이터 로드
tsll = pd.read_csv('/content/data/raw/TSLL_daily.csv', index_col=0, parse_dates=True)
tsla = pd.read_csv('/content/data/raw/TSLA_daily.csv', index_col=0, parse_dates=True)

# 필요한 컬럼만
for df in [tsll, tsla]:
    df.index = pd.to_datetime(df.index)

print(f'TSLL 로드: {len(tsll)}행')
print(f'TSLA 로드: {len(tsla)}행')

In [ ]:
# ============================================================
# 셀 2: 기술적 지표 계산 함수
#
# pandas_ta 라이브러리: 100가지 이상의 기술적 지표를 한 줄로 계산
# 직접 수식을 구현할 필요 없이 검증된 라이브러리 사용
# ============================================================

def add_technical_indicators(df):
    """OHLCV 데이터프레임에 기술적 지표 추가"""
    df = df.copy()
    
    # --- 추세 지표 ---
    # SMA: Simple Moving Average (단순 이동평균)
    # 최근 N일 종가의 평균 → 추세 방향 파악
    df['sma_20'] = ta.sma(df['Close'], length=20)
    df['sma_50'] = ta.sma(df['Close'], length=50)
    
    # EMA: Exponential Moving Average (지수 이동평균)
    # 최근 데이터에 더 높은 가중치 → 추세 전환에 더 민감
    df['ema_12'] = ta.ema(df['Close'], length=12)
    df['ema_26'] = ta.ema(df['Close'], length=26)
    
    # 가격이 이동평균 대비 위치 (정규화)
    df['price_vs_sma20'] = (df['Close'] - df['sma_20']) / df['sma_20']
    df['price_vs_sma50'] = (df['Close'] - df['sma_50']) / df['sma_50']
    
    # --- 모멘텀 지표 ---
    # RSI: Relative Strength Index (상대강도지수)
    # 0~100 사이의 값 / 70 이상: 과매수(매도 신호) / 30 이하: 과매도(매수 신호)
    df['rsi_14'] = ta.rsi(df['Close'], length=14)
    df['rsi_7']  = ta.rsi(df['Close'], length=7)   # 단기 RSI
    
    # MACD: Moving Average Convergence Divergence
    # EMA(12) - EMA(26) = MACD선 / 시그널선(MACD의 9일 EMA)과의 교차가 매매 신호
    macd = ta.macd(df['Close'], fast=12, slow=26, signal=9)
    if macd is not None:
        df['macd']        = macd.iloc[:, 0]   # MACD선
        df['macd_signal'] = macd.iloc[:, 1]   # 시그널선
        df['macd_hist']   = macd.iloc[:, 2]   # 히스토그램 (MACD - Signal)
    
    # --- 변동성 지표 (스윙매매 핵심!) ---
    # Bollinger Bands: 이동평균 ± 2*표준편차
    # 상단 밴드 근처 = 과매수 / 하단 밴드 근처 = 과매도
    bb = ta.bbands(df['Close'], length=20, std=2)
    if bb is not None:
        df['bb_upper']  = bb.iloc[:, 0]   # 상단 밴드
        df['bb_mid']    = bb.iloc[:, 1]   # 중간 밴드 (SMA20)
        df['bb_lower']  = bb.iloc[:, 2]   # 하단 밴드
        df['bb_width']  = (df['bb_upper'] - df['bb_lower']) / df['bb_mid']  # 밴드 폭 (변동성 지수)
        df['bb_pct']    = (df['Close'] - df['bb_lower']) / (df['bb_upper'] - df['bb_lower'])  # 밴드 내 위치
    
    # ATR: Average True Range (평균 실제 범위)
    # 하루의 실제 가격 변동 범위 평균
    # 내일 예상 고가 = 종가 + ATR * 계수 / 예상 저가 = 종가 - ATR * 계수
    # => 스윙매매 목표가 설정의 핵심 지표!
    df['atr_14'] = ta.atr(df['High'], df['Low'], df['Close'], length=14)
    df['atr_7']  = ta.atr(df['High'], df['Low'], df['Close'], length=7)
    df['atr_pct'] = df['atr_14'] / df['Close']  # ATR의 가격 대비 비율 (정규화)
    
    # --- 거래량 지표 ---
    # OBV: On-Balance Volume
    # 상승일엔 거래량 +, 하락일엔 - 누적 → 스마트머니 방향 파악
    df['obv'] = ta.obv(df['Close'], df['Volume'])
    df['obv_ema'] = ta.ema(df['obv'], length=20)  # OBV의 20일 EMA
    
    # 거래량 비율 (오늘 거래량 / 20일 평균 거래량)
    df['volume_ratio'] = df['Volume'] / df['Volume'].rolling(20).mean()
    
    # --- 가격 패턴 특성 ---
    # 전일 대비 수익률
    df['return_1d'] = df['Close'].pct_change(1)
    df['return_3d'] = df['Close'].pct_change(3)
    df['return_5d'] = df['Close'].pct_change(5)
    df['return_10d'] = df['Close'].pct_change(10)
    
    # Lag 특성 (어제의 데이터)
    for lag in [1, 2, 3, 5]:
        df[f'close_lag{lag}'] = df['Close'].shift(lag)
        df[f'return_lag{lag}'] = df['return_1d'].shift(lag)
        df[f'high_lag{lag}']  = df['High'].shift(lag)
        df[f'low_lag{lag}']   = df['Low'].shift(lag)
    
    # 캘린더 특성
    df['day_of_week'] = df.index.dayofweek
    df['month'] = df.index.month
    df['week_of_year'] = df.index.isocalendar().week.astype(int)
    
    # 일중 특성
    df['hl_ratio']   = df['High'] / df['Low']        # 당일 고저 비율
    df['co_ratio']   = df['Close'] / df['Open']      # 종가/시가 비율 (당일 방향성)
    df['intraday_range'] = (df['High'] - df['Low']) / df['Open']
    
    return df

print('피처 엔지니어링 함수 정의 완료')

In [ ]:
# ============================================================
# 셀 3: TSLL과 TSLA에 지표 적용 + 데이터 병합
# ============================================================

# TSLL 지표 계산
tsll_feat = add_technical_indicators(tsll)

# TSLA 지표도 계산 (TSLL 예측에 도움이 되는 TSLA 지표만 추가)
tsla_feat = add_technical_indicators(tsla)
tsla_cols = ['Close', 'return_1d', 'rsi_14', 'atr_pct', 'macd_hist', 'volume_ratio']
tsla_sub = tsla_feat[[c for c in tsla_cols if c in tsla_feat.columns]].copy()
tsla_sub.columns = [f'tsla_{c}' for c in tsla_sub.columns]

# 병합
combined = tsll_feat.join(tsla_sub, how='left')

# --- 예측 타겟(Target) 변수 생성 ---
# 우리가 예측하려는 것: 내일의 고가와 저가
# shift(-1): 오늘 행에 내일 데이터를 붙임 (학습 데이터 구성)
combined['target_high']  = combined['High'].shift(-1)   # 내일 고가
combined['target_low']   = combined['Low'].shift(-1)    # 내일 저가
combined['target_close'] = combined['Close'].shift(-1)  # 내일 종가 (참고용)
combined['target_return'] = combined['return_1d'].shift(-1)  # 내일 수익률

# 결측치 제거 (이동평균 등 계산에 필요한 초기 데이터 + 마지막 1행)
combined.dropna(inplace=True)

print(f'최종 데이터셋: {len(combined)}행 x {combined.shape[1]}열')
print(f'날짜 범위: {combined.index[0].date()} ~ {combined.index[-1].date()}')
print(f'\n특성 목록 ({combined.shape[1]}개):')
print([c for c in combined.columns if not c.startswith('target')])

In [ ]:
# ============================================================
# 셀 4: 데이터 분할 (학습/검증/테스트)
#
# 주식 데이터는 시간 순서가 있습니다!
# 일반 ML처럼 랜덤 분할 하면 안 됩니다.
# => 미래 데이터가 학습에 들어가는 "데이터 누수(Data Leakage)" 발생
#
# 올바른 분할:
# [학습 60%] [검증 20%] [테스트 20%] — 시간순
# ============================================================

n = len(combined)
train_end = int(n * 0.70)
val_end   = int(n * 0.85)

train_df = combined.iloc[:train_end]
val_df   = combined.iloc[train_end:val_end]
test_df  = combined.iloc[val_end:]

print('데이터 분할 결과:')
print(f'  학습셋:  {len(train_df)}행 ({train_df.index[0].date()} ~ {train_df.index[-1].date()})')
print(f'  검증셋:  {len(val_df)}행  ({val_df.index[0].date()} ~ {val_df.index[-1].date()})')
print(f'  테스트셋: {len(test_df)}행  ({test_df.index[0].date()} ~ {test_df.index[-1].date()})')

# 시각화
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(train_df.index, train_df['Close'], label=f'학습 ({len(train_df)}일)', color='#26a69a')
ax.plot(val_df.index,   val_df['Close'],   label=f'검증 ({len(val_df)}일)',  color='#ffd700')
ax.plot(test_df.index,  test_df['Close'],  label=f'테스트 ({len(test_df)}일)', color='#ef5350')
ax.axvline(train_df.index[-1], color='white', linestyle='--', linewidth=1)
ax.axvline(val_df.index[-1],   color='white', linestyle='--', linewidth=1)
ax.set_title('TSLL 데이터 분할 (시간 순서 유지)')
ax.set_ylabel('가격 (USD)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/content/data_split.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# 셀 5: 특성 중요도 미리보기 (상관관계 히트맵)
# ============================================================

# 핵심 특성과 타겟의 상관관계
key_features = ['rsi_14', 'macd_hist', 'bb_pct', 'atr_pct', 'volume_ratio',
                'return_1d', 'return_3d', 'return_5d', 'price_vs_sma20',
                'tsla_return_1d', 'tsla_rsi_14']
targets = ['target_high', 'target_low', 'target_close']

avail_feats = [f for f in key_features if f in combined.columns]
avail_targs = [t for t in targets if t in combined.columns]

corr_matrix = combined[avail_feats + avail_targs].corr()
target_corr = corr_matrix.loc[avail_feats, avail_targs]

fig, ax = plt.subplots(figsize=(8, 10))
import matplotlib.colors as mcolors
cmap = plt.cm.RdYlGn
im = ax.imshow(target_corr.values, cmap=cmap, vmin=-0.5, vmax=0.5, aspect='auto')
plt.colorbar(im, ax=ax)
ax.set_xticks(range(len(avail_targs)))
ax.set_yticks(range(len(avail_feats)))
ax.set_xticklabels(avail_targs, rotation=30)
ax.set_yticklabels(avail_feats)
for i in range(len(avail_feats)):
    for j in range(len(avail_targs)):
        ax.text(j, i, f'{target_corr.iloc[i, j]:.2f}', ha='center', va='center', fontsize=8)
ax.set_title('특성-타겟 상관관계')
plt.tight_layout()
plt.savefig('/content/feature_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# 셀 6: 데이터 저장
# ============================================================

os.makedirs('/content/data/processed', exist_ok=True)

combined.to_csv('/content/data/processed/TSLL_features.csv')
train_df.to_csv('/content/data/processed/train.csv')
val_df.to_csv('/content/data/processed/val.csv')
test_df.to_csv('/content/data/processed/test.csv')

print('저장 완료!')
print(f'전체 특성 수: {combined.shape[1] - 3}개 (타겟 3개 제외)')
print('\n다음 단계: 03_models_arima_prophet.ipynb')
print('=> ARIMA, Prophet 모델 학습 시작')

## Phase 2 완료

### 만들어진 주요 특성
- **ATR**: 예상 고가/저가 범위 계산의 핵심
- **RSI**: 과매수/과매도 신호 → 반전 예측
- **Bollinger Band %**: 현재 가격이 밴드 어디에 위치하는지
- **MACD Histogram**: 모멘텀 변화 감지
- **TSLA 지표**: TSLL 예측의 강력한 보조 변수

### 다음에 할 것
3번 노트북: ARIMA와 Prophet으로 기준선(Baseline) 성능 측정